# Model card data: pull a run from `mlflow.db`, load its checkpoint, measure it

Looks up runs by experiment name in `mlflow.db` (sqlite), matches the run to its checkpoint in `models_checkpoints/` (checkpoints are named `{model}-epoch={n}-val_acc={acc}-{run_id[:8]}.ckpt`), rebuilds the architecture from the logged `pretrained_model` param via `PRETRAINED_MODEL_REGISTRY`, and reports size / latency / other model-card fields.

In [48]:
import sqlite3
import sys
import time
from pathlib import Path

import torch

BASE_DIR = Path.cwd().parent
sys.path.insert(0, str(BASE_DIR))
sys.path.insert(0, str(BASE_DIR / "train"))

from train.config import PRETRAINED_MODEL_REGISTRY  # noqa: E402
from src.classifier import FlowerClassifier  # noqa: E402
from src.data import FlowerDataset  # noqa: E402

DB_PATH = BASE_DIR / "mlflow.db"
CKPT_DIR = BASE_DIR / "models_checkpoints"
EXP_NAME = "flower-classification-v2"

## Pull runs for the experiment

In [49]:
def get_runs(exp_name: str) -> list[dict]:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        exp = conn.execute(
            "SELECT experiment_id FROM experiments WHERE name = ?", (exp_name,)
        ).fetchone()
        if exp is None:
            raise ValueError(f"No experiment named {exp_name!r} in {DB_PATH}")

        run_rows = conn.execute(
            "SELECT run_uuid, name, status, start_time, end_time FROM runs "
            "WHERE experiment_id = ? ORDER BY start_time DESC",
            (exp["experiment_id"],),
        ).fetchall()

        runs = []
        for r in run_rows:
            params = dict(
                conn.execute(
                    "SELECT key, value FROM params WHERE run_uuid = ?", (r["run_uuid"],)
                ).fetchall()
            )
            metrics = dict(
                conn.execute(
                    "SELECT key, value FROM latest_metrics WHERE run_uuid = ?",
                    (r["run_uuid"],),
                ).fetchall()
            )
            runs.append({**dict(r), "params": params, "metrics": metrics})
        return runs
    finally:
        conn.close()


runs = get_runs(EXP_NAME)
print(f"{len(runs)} run(s) for experiment {EXP_NAME!r}")
for r in runs:
    print(r["run_uuid"][:8], r["params"].get("pretrained_model"), "val_acc =", r["metrics"].get("val_acc"))

71 run(s) for experiment 'flower-classification-v2'
d6508546 efficientnet_v2_s val_acc = 0.9996511340141296
33cfe211 efficientnet_b0 val_acc = 0.9872667193412781
c5c91a63 efficientnet_b0 val_acc = 0.08477237075567245
8d080eb0 efficientnet_b0 val_acc = 0.914878785610199
e6e0fc8b efficientnet_v2_s val_acc = 0.9935461282730103
9dc68920 mobilenet_v3_large val_acc = 0.992499589920044
9d3d7993 efficientnet_b0 val_acc = 0.9037153124809265
84508b62 efficientnet_b1 val_acc = 0.9618000984191895
b9224378 mobilenet_v3_large val_acc = 0.9996511340141296
21adfe74 efficientnet_b0 val_acc = 0.9947671294212341
05b8075c efficientnet_v2_s val_acc = 0.9993022680282593
717ad714 resnet50 val_acc = 0.7012035846710205
bec22111 efficientnet_b2 val_acc = 0.992499589920044
0c4a9681 efficientnet_b2 val_acc = 0.9895342588424683
28ea350e efficientnet_b2 val_acc = 0.9898831248283386
cfe14657 resnet50 val_acc = 0.942438542842865
1d95d952 efficientnet_b1 val_acc = 0.9965113997459412
4632c1bb convnext_tiny val_acc = 0.

## Pick a run and find its checkpoint

In [50]:
def find_checkpoint(run_uuid: str) -> Path:
    matches = list(CKPT_DIR.glob(f"*-{run_uuid[:8]}.ckpt"))
    if not matches:
        raise FileNotFoundError(f"No checkpoint in {CKPT_DIR} for run {run_uuid[:8]}")
    return matches[0]


# pick best val_acc by default; swap for runs[i] to target a specific run
run = max(runs, key=lambda r: float(r["metrics"].get("val_acc", 0)))
ckpt_path = find_checkpoint(run["run_uuid"])

print("run:", run["run_uuid"])
print("checkpoint:", ckpt_path)
print("params:", run["params"])
print("metrics:", run["metrics"])

run: 6899e8e2c48e429c846ca3428c248753
checkpoint: /home/zelluzy/Desktop/code/flowers/models_checkpoints/vit_b_16-epoch=18-val_acc=1.000-6899e8e2.ckpt
params: {'effective_batch_size': '256', 'exp_name': 'flower-classification-v2', 'mlflow_db_uri': 'sqlite:////home/zelluzy/Desktop/code/flowers/mlflow.db', 'lr_head_stage_1': '0.001', 'lr_head_stage_2': '0.001', 'lr_backbone': '1e-05', 'unfreeze_at_epoch': '5', 'max_epochs': '50', 'batch_size': '64', 'accumulate_grad_batches': '4', 'early_stopping_patience': '5', 'precision': '16-mixed', 'optimizer': 'adamw', 'scheduler': 'cosine', 'pretrained_model': 'vit_b_16', 'optimizer_kwargs/weight_decay': '0.01', 'scheduler_kwargs/T_max': '50', 'scheduler_kwargs/eta_min': '1e-06', 'lr_head': '0.001', 'num_classes': '102', 'weight_decay': '0.01', 'head_name': 'heads'}
metrics: {'lr-AdamW': 0.0009843072889837512, 'train_loss_step': 0.014805328100919724, 'train_acc_step': 1.0, 'train_f1': 0.9999999403953552, 'epoch': 23.0, 'val_loss': 0.008795605041086

## Rebuild the architecture and load checkpoint weights

In [ ]:
pretrained_model_name = run["params"]["pretrained_model"]
factory, head_name, backbone_name = PRETRAINED_MODEL_REGISTRY[pretrained_model_name]

dataset = FlowerDataset(BASE_DIR / "data")
num_classes = len(dataset.classes)

model = FlowerClassifier.load_from_checkpoint(
    ckpt_path,
    pretrained_model=factory(),
    num_classes=num_classes,
    class_weights=None,
    head_name=head_name,
    class_names=dataset.classes,
    map_location="cpu",
    strict=False,  # checkpoint has a criterion.weight buffer we don't restore for inference
)
model.eval()
print(model.__class__.__name__, "loaded on", next(model.parameters()).device)

FlowerClassifier loaded on cpu


/home/zelluzy/Desktop/code/flowers/.venv/lib/python3.13/site-packages/lightning/pytorch/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['criterion.weight']


In [52]:
print(model.device)

cpu


## Model card metrics: size and latency

In [53]:
def model_card_info(model: torch.nn.Module, ckpt_path: Path, input_size=(1, 3, 224, 224), n_warmup=5, n_runs=30) -> dict:
    n_params = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())

    x = torch.randn(*input_size, device=next(model.parameters()).device)
    with torch.no_grad():
        for _ in range(n_warmup):
            model(x)

        latencies = []
        for _ in range(n_runs):
            start = time.perf_counter()
            model(x)
            latencies.append((time.perf_counter() - start) * 1000)

    latencies.sort()
    return {
        "num_parameters": n_params,
        "num_trainable_parameters": n_trainable,
        "model_size_mb": (param_bytes + buffer_bytes) / 1e6,  # weights + buffers, i.e. a deployable state_dict
        "checkpoint_size_mb": ckpt_path.stat().st_size / 1e6,  # full .ckpt: also has optimizer/scheduler state
        "input_size": input_size,
        "latency_ms_mean": sum(latencies) / len(latencies),
        "latency_ms_p50": latencies[len(latencies) // 2],
        "latency_ms_p95": latencies[int(len(latencies) * 0.95)],
        "device": str(next(model.parameters()).device),
    }


info = model_card_info(model, ckpt_path)
for k, v in info.items():
    print(f"{k:>28}: {v}")

              num_parameters: 85877094
    num_trainable_parameters: 85877094
               model_size_mb: 343.508784
          checkpoint_size_mb: 1030.723581
                  input_size: (1, 3, 224, 224)
             latency_ms_mean: 90.0204905667124
              latency_ms_p50: 88.76342800067505
              latency_ms_p95: 102.15778299971134
                      device: cpu


## Remaining model-card fields (architecture, data, training run)

In [54]:
model_card = {
    "architecture": pretrained_model_name,
    "head_name": head_name,
    "num_classes": num_classes,
    "class_names": dataset.classes,
    "mlflow_run_id": run["run_uuid"],
    "mlflow_experiment": EXP_NAME,
    "checkpoint_file": ckpt_path.name,
    "training_params": run["params"],
    "training_metrics": run["metrics"],
    **info,
}
model_card

{'architecture': 'vit_b_16',
 'head_name': 'heads',
 'num_classes': 102,
 'class_names': [" 'pink primrose'",
  " 'hard-leaved pocket orchid'",
  " 'canterbury bells'",
  " 'sweet pea'",
  " 'english marigold'",
  " 'tiger lily'",
  " 'moon orchid'",
  " 'bird of paradise'",
  " 'monkshood'",
  " 'globe thistle'",
  " 'snapdragon'",
  ' "colt\'s foot"',
  " 'king protea'",
  " 'spear thistle'",
  " 'yellow iris'",
  " 'globe-flower'",
  " 'purple coneflower'",
  " 'peruvian lily'",
  " 'balloon flower'",
  " 'giant white arum lily'",
  " 'fire lily'",
  " 'pincushion flower'",
  " 'fritillary'",
  " 'red ginger'",
  " 'grape hyacinth'",
  " 'corn poppy'",
  " 'prince of wales feathers'",
  " 'stemless gentian'",
  " 'artichoke'",
  " 'sweet william'",
  " 'carnation'",
  " 'garden phlox'",
  " 'love in the mist'",
  " 'mexican aster'",
  " 'alpine sea holly'",
  " 'ruby-lipped cattleya'",
  " 'cape flower'",
  " 'great masterwort'",
  " 'siam tulip'",
  " 'lenten rose'",
  " 'barbeton 

## Compare pinned runs

In [55]:
PINNED_RUN_IDS = [
    "6899e8e2c48e429c846ca3428c248753",
    "4632c1bb35b4448f9584b3d4f164b8ed",
    "747435c3c8a64db3a2ac488cceff2b89",
    "d650854670754e90a355b4adc7e450e3",
    # "785424a8e99c400bb68e6fbd7cb76e14",
]


def get_run(run_uuid: str) -> dict:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        r = conn.execute(
            "SELECT run_uuid, name, status, start_time, end_time FROM runs WHERE run_uuid = ?",
            (run_uuid,),
        ).fetchone()
        if r is None:
            raise ValueError(f"No run {run_uuid!r} in {DB_PATH}")
        params = dict(
            conn.execute(
                "SELECT key, value FROM params WHERE run_uuid = ?", (run_uuid,)
            ).fetchall()
        )
        metrics = dict(
            conn.execute(
                "SELECT key, value FROM latest_metrics WHERE run_uuid = ?", (run_uuid,)
            ).fetchall()
        )
        return {**dict(r), "params": params, "metrics": metrics}
    finally:
        conn.close()


def load_model_for_run(run: dict) -> tuple[FlowerClassifier, Path]:
    ckpt = find_checkpoint(run["run_uuid"])
    name = run["params"]["pretrained_model"]
    factory, head_name, _ = PRETRAINED_MODEL_REGISTRY[name]
    m = FlowerClassifier.load_from_checkpoint(
        ckpt,
        pretrained_model=factory(),
        num_classes=num_classes,
        class_weights=None,
        head_name=head_name,
        class_names=dataset.classes,
        map_location="cpu",
        strict=False,
    )
    m.eval()
    return m, ckpt


pinned_runs = [get_run(rid) for rid in PINNED_RUN_IDS]

In [56]:
import pandas as pd

rows = []
for r in pinned_runs:
    m, ckpt = load_model_for_run(r)
    info = model_card_info(m, ckpt)
    rows.append(
        {
            "run_id": r["run_uuid"][:8],
            "architecture": r["params"].get("pretrained_model"),
            "val_acc": r["metrics"].get("val_acc"),
            "val_f1": r["metrics"].get("val_f1"),
            "num_parameters": info["num_parameters"],
            "model_size_mb": info["model_size_mb"],
            "checkpoint_size_mb": info["checkpoint_size_mb"],
            "latency_ms_mean": info["latency_ms_mean"],
            "latency_ms_p95": info["latency_ms_p95"],
        }
    )
    del m  # free memory before loading the next model

comparison = pd.DataFrame(rows).set_index("run_id")
comparison

,architecture,val_acc,val_f1,num_parameters,model_size_mb,checkpoint_size_mb,latency_ms_mean,latency_ms_p95
run_id,,,,,,,,
6899e8e2,vit_b_16,1.000000,1.000000,85877094,343.508784,1030.723581,84.234142,88.185703
4632c1bb,convnext_tiny,0.999826,0.999831,27898566,111.594672,335.018119,29.638832,38.634164
747435c3,resnet50,0.999651,0.999521,23717030,95.081432,285.072689,31.314553,33.576074
d6508546,efficientnet_v2_s,0.999651,0.999472,20308150,81.849376,245.016249,30.861859,32.389114


In [57]:
comparison.to_csv("best_models_comparison.csv")
